# 02 — BQML Forecast Model

Create and evaluate the ARIMA+ time-series forecast model for cash flow prediction.

In [ ]:
import os
from google.cloud import bigquery

PROJECT_ID = os.environ.get('PROJECT_ID', 'your-project-id')
DATASET_ID = 'cash_agent_demo'
client = bigquery.Client(project=PROJECT_ID)

## Step 1: Prepare Training Data

Daily net cash flow by currency from the cash journal.

In [ ]:
query = f"""
SELECT posting_date, currency,
       SUM(CASE WHEN transaction_type='INFLOW' THEN amount ELSE -amount END) AS net_cash_flow
FROM `{PROJECT_ID}.{DATASET_ID}.cash_journal`
GROUP BY posting_date, currency
ORDER BY currency, posting_date
"""
df = client.query(query).to_dataframe()
print(f"Training data: {len(df)} rows")
print(f"Date range: {df.posting_date.min()} to {df.posting_date.max()}")
print(f"Currencies: {df.currency.unique().tolist()}")
df.groupby('currency')['net_cash_flow'].describe()

## Step 2: Create ARIMA+ Model

This creates a multi-series ARIMA+ model with automatic parameter selection.

In [ ]:
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.cash_forecast_model`
OPTIONS(
    model_type = 'ARIMA_PLUS',
    time_series_timestamp_col = 'posting_date',
    time_series_data_col = 'net_cash_flow',
    time_series_id_col = 'currency',
    horizon = 90,
    auto_arima = TRUE
) AS
SELECT posting_date, currency,
       SUM(CASE WHEN transaction_type='INFLOW' THEN amount ELSE -amount END) AS net_cash_flow
FROM `{PROJECT_ID}.{DATASET_ID}.cash_journal`
GROUP BY posting_date, currency
"""

print("Creating BQML ARIMA+ model... (this may take 2-5 minutes)")
job = client.query(create_model_sql)
job.result()
print("Model created successfully!")

## Step 3: Evaluate Model

In [ ]:
eval_sql = f"""
SELECT *
FROM ML.ARIMA_EVALUATE(MODEL `{PROJECT_ID}.{DATASET_ID}.cash_forecast_model`)
"""
client.query(eval_sql).to_dataframe()

## Step 4: Generate 30-Day Forecast

In [ ]:
forecast_sql = f"""
SELECT
    forecast_timestamp AS forecast_date,
    forecast_value AS net_cash_flow,
    standard_error,
    prediction_interval_lower_bound AS lower_bound,
    prediction_interval_upper_bound AS upper_bound,
    currency
FROM ML.FORECAST(
    MODEL `{PROJECT_ID}.{DATASET_ID}.cash_forecast_model`,
    STRUCT(30 AS horizon, 0.95 AS confidence_level)
)
ORDER BY currency, forecast_timestamp
"""
forecast_df = client.query(forecast_sql).to_dataframe()
print(f"Forecast rows: {len(forecast_df)}")
forecast_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
for i, ccy in enumerate(['USD', 'EUR', 'GBP']):
    ccy_df = forecast_df[forecast_df.currency == ccy]
    ax = axes[i]
    ax.plot(ccy_df.forecast_date, ccy_df.net_cash_flow, label='Forecast', color='#0070F2')
    ax.fill_between(ccy_df.forecast_date, ccy_df.lower_bound, ccy_df.upper_bound,
                    alpha=0.2, color='#0070F2', label='95% CI')
    ax.set_title(f'{ccy} Daily Net Cash Flow Forecast')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()